### 1. Environment set up

In [1]:
# Auto realoading "magic" for IPython
# Load the autoreload extension
%load_ext autoreload 
# Automatically reload all modules (except those excluded by %aimport) every time before executing the Python code typed.
%autoreload 2 

In [7]:
# Import necessary modules
import sys # For manipulating the Python path
from pathlib import Path # For handling filesystem paths in an object-oriented way
import pandas as pd # For data manipulation and analysis

# If the notebook is in "notebooks/", the parent is the Project Root.
# Path.cwd() <- Current Working Directory
project_root = Path.cwd().parent 

# This ensures Python looks here FIRST, before looking in installed packages.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root)) # Convert Path object to string for sys.path
    print(f"Added to sys.path: {project_root}")

print("cwd:", Path.cwd())
print("project_root:", project_root)

from src.load_data import load_raw_data
from src.clean_data import clean_dataframe

cwd: /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/notebooks
project_root: /home/idiazl/2026_MLOps/2. mlops_modulirized-repo


### 2. Pipeline configuration

In [3]:
RAW_DATA_PATH = project_root / "data" / "raw" / "opiod_raw_data.csv"
TARGET_COLUMN = "OD"

print("Raw data path:", RAW_DATA_PATH)
print("Target column:", TARGET_COLUMN)

Raw data path: /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/data/raw/opiod_raw_data.csv
Target column: OD


### 3. Execute load and clean steps (modules)

In [5]:
df_raw = load_raw_data(RAW_DATA_PATH)
df_clean = clean_dataframe(df_raw, target_column=TARGET_COLUMN)

print("df_raw.shape:", df_raw.shape)
print("df_clean.shape:", df_clean.shape)

[load_data.load_raw_data] Loading raw data from /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/data/raw/opiod_raw_data.csv
[utils.load_csv] Loading CSV from /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/data/raw/opiod_raw_data.csv
[load_data.load_raw_data] Loaded dataframe shape: (1000, 22)
[clean_data.clean_dataframe] Cleaning dataframe
[clean_data.clean_dataframe] Dropped 102 rows due to NA or duplicates
[clean_data.clean_dataframe] Rows after cleaning: 898
df_raw.shape: (1000, 22)
df_clean.shape: (898, 21)


### ---- MLOps verfication ---

In [ ]:
# 1) Column normalization proof
raw_has_rx_ds = "rx ds" in df_raw.columns
clean_has_rx_ds = "rx_ds" in df_clean.columns

print('Raw has "rx ds":', raw_has_rx_ds)
print('Clean has "rx_ds":', clean_has_rx_ds)

if raw_has_rx_ds:
    assert clean_has_rx_ds, 'Expected "rx ds" to be normalized to "rx_ds"'

# 2) Index integrity proof
index_is_contiguous = df_clean.index.equals(pd.RangeIndex(len(df_clean)))
print("Index reset and contiguous:", index_is_contiguous)
assert index_is_contiguous, "Expected RangeIndex(0..n-1) after reset_index(drop=True)"

# 3) Target presence proof
target_present = TARGET_COLUMN in df_clean.columns
print(f"Target '{TARGET_COLUMN}' preserved:", target_present)
assert target_present, f"Missing target column after cleaning: {TARGET_COLUMN}"

df_clean.head()

--- MLOps Verification ---
Raw has "rx ds": True
Clean has "rx_ds": True
Index reset and contiguous: True
Target 'OD' preserved: True


,OD,Low_inc,SURG,rx_ds,A,B,C,D,E,F,...,I,J,K,L,M,N,R,S,T,V
0,1,1,0,330,0,0,0,1,1,1,...,1,1,0,1,1,1,1,0,0,0
1,1,1,0,457,0,0,1,0,1,0,...,1,1,0,0,1,1,1,1,0,0
2,1,1,1,722,0,1,0,1,1,0,...,1,1,1,0,1,0,1,0,1,0
3,0,1,0,262,0,1,0,0,1,1,...,1,0,1,1,1,1,1,1,1,0
4,1,1,0,780,0,0,0,0,1,1,...,1,0,0,0,1,0,1,0,0,0


In [ ]:
# Column Comparison (first 20)

raw_cols = list(df_raw.columns)
clean_cols = list(df_clean.columns)

print("\nRaw columns:")
print(raw_cols[:20])

print("\nClean columns:")
print(clean_cols[:20])

# Optional structured diff for teaching
removed_cols = sorted(set(raw_cols) - set(clean_cols))
added_cols = sorted(set(clean_cols) - set(raw_cols))

print("\nColumns removed during cleaning:")
print(removed_cols[:10])

print("\nColumns added during cleaning:")
print(added_cols[:10])


Raw columns:
['ID', 'OD', 'Low_inc', 'SURG', 'rx ds', 'A', 'B', 'C', 'D', 'E', 'F', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'R', 'S']

Clean columns:
['OD', 'Low_inc', 'SURG', 'rx_ds', 'A', 'B', 'C', 'D', 'E', 'F', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'R', 'S', 'T']

Columns removed during cleaning:
['ID', 'rx ds']

Columns added during cleaning:
['rx_ds']
